# Fitness App Churn Project

In this project, we assume we have a Fitness Gym business; the retention team would like to know, given its user data, and deliver them a table of churn probabilities and expected revenue at risk given the various subscription prices: Basic - $\$9.99$ per month; Premium - $\$19.99$ per month; and Elite - $\$29.99$ per month.

"Churn" means a customer cancelling or stopping their subscription — leaving the service entirely.

The word comes from the idea of customers "churning" in and out of a business — like butter being churned, constantly turning over. It's used broadly across subscription businesses: streaming services (Netflix), SaaS products, gyms, phone plans, subscription boxes — anywhere revenue depends on people staying subscribed month over month rather than making a one-time purchase.

**Why it matters so much to companies:** 

Acquiring a new customer is typically far more expensive than retaining an existing one (ad spend, onboarding costs, etc.), so a small reduction in churn rate can have an outsized impact on revenue. That's exactly why "churn prediction" is such a common real-world Data Scientist project.

What this project does:

Build the complete pipeline: SQL feature engineering → PyTorch model → business-ready output.

Using the following skills:

`SQL`: filtering, aggregating, joins (inner and left), CTEs, subqueries, window functions, and a self-join for rolling windows — assembled into a real feature-engineering query\
`Pandas`: bridging SQL output into one-hot encoded, properly typed numeric data\
`PyTorch`: tensors, a neural network with Linear/ReLU/Dropout, a training loop with gradients and backpropagation, class-imbalance handling, and proper train/test evaluation\
`Business translation`: converting a probability into an actual prioritised dollar-value output

We have a SQLite database: `fitness_app.db` with 5 tables:

`users(user_id, signup_date, plan_type, country, acquisition_channel)`

`subscriptions(subscription_id, user_id, start_date, end_date, monthly_price, status)`

`workouts(workout_id, user_id, workout_date, duration_minutes, workout_type, calories_burned)`

`app_events(event_id, user_id, event_date, event_type)` — event_type ∈ {login, push_notification_open, support_ticket, plan_upgrade, plan_downgrade}

`payments(payment_id, user_id, payment_date, amount, payment_status)` — payment_status ∈ {success, failed}

## The Churn Feature Table (SQL query)

Here's the target output:

user_id\
plan_type, country, acquisition_channel\
tenure_days (signup to today)\
total_workouts\
avg_duration_minutes\
workouts_last_30d\
days_since_last_login\
failed_payments_last_90d\
support_tickets_total\
label (1 if cancelled, else 0)

The strategy — don't write this as one giant query. Instead, build a separate CTE for each group of features, then join them all together at the end.

In [2]:
import sqlite3, pandas as pd
conn = sqlite3.connect("fitness_app.db")

### First Common Table Expression (CTE) — workout stats:

For workouts_last_30d the "last 30 days" needs to be relative to some reference date. For a live system, that'd be "today". Since our dataset is historical, we're treating 2025-08-01 as the reference "today" for the whole project (that's the end of the data window in fitness_app.db).

In [3]:
pd.read_sql_query("""WITH workout_stats AS (
    SELECT
        user_id,
        COUNT(user_id) AS total_workouts,
        AVG(duration_minutes) AS avg_duration_minutes,
        SUM(CASE WHEN workout_date > date('2025-08-01', '-30 days') THEN 1 ELSE 0 END) AS workouts_last_30d
    FROM workouts
    GROUP BY user_id
)
SELECT * FROM workout_stats""", conn)

,user_id,total_workouts,avg_duration_minutes,workouts_last_30d
0,2,25,32.120000,0
1,3,56,35.589286,9
2,4,3,27.333333,0
3,5,44,31.090909,6
4,6,50,34.580000,3
...,...,...,...,...
731,795,33,34.787879,1
732,796,19,31.947368,0
733,798,1,37.000000,0
734,799,17,31.647059,0


Missing 64 of the users who never worked out.

Why COUNT(*) instead of COUNT(user_id)?

Some people prefer COUNT(\*) because it clearly signals "I'm counting rows," others prefer COUNT(specific_column) because it's explicit about intent. COUNT(column) becomes meaningfully different from COUNT(*) the moment that column can contain NULLs (it would then undercount)

### Second CTE - login stats

In [4]:
pd.read_sql_query("""WITH login_stats AS (
    SELECT
        user_id,
        MAX(event_date) AS last_login_date
    FROM app_events
    WHERE event_type = 'login'
    GROUP BY user_id
)
SELECT * FROM login_stats""", conn)

,user_id,last_login_date
0,1,2024-10-19
1,2,2025-07-23
2,3,2025-07-25
3,4,2025-03-14
4,5,2025-07-31
...,...,...
790,796,2025-04-16
791,797,2025-06-10
792,798,2024-07-31
793,799,2025-08-01


### Third CTE - payment stats

In [5]:
pd.read_sql_query("""WITH payment_stats AS (
    SELECT 
        user_id,
        SUM(CASE WHEN payment_status = 'failed' 
                      AND payment_date > date('2025-08-01', '-90 days') 
            THEN 1 ELSE 0 END) AS failed_payments_last_90d
    FROM payments
    GROUP BY user_id
)
SELECT * FROM payment_stats""", conn)

,user_id,failed_payments_last_90d
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
...,...,...
795,796,0
796,797,0
797,798,0
798,799,0


### Fourth CTE - ticket stats

In [6]:
pd.read_sql_query("""WITH ticket_stats AS (
    SELECT 
        user_id,
        COUNT(user_id) AS support_ticket_total
    FROM app_events
    WHERE event_type='support_ticket'
    GROUP BY user_id
)
SELECT * FROM ticket_stats""", conn)

,user_id,support_ticket_total
0,2,2
1,3,2
2,4,1
3,5,1
4,6,1
...,...,...
563,793,1
564,794,2
565,795,2
566,796,2


### Left-JOIN all CTEs to `users` table

In [7]:
pd.read_sql_query("""
WITH workout_stats AS (
    SELECT
        user_id,
        COUNT(user_id) AS total_workouts,
        AVG(duration_minutes) AS avg_duration_minutes,
        SUM(CASE WHEN workout_date > date('2025-08-01', '-30 days') THEN 1 ELSE 0 END) AS workouts_last_30d
    FROM workouts
    GROUP BY user_id
),
login_stats AS (
    SELECT
        user_id,
        MAX(event_date) AS last_login_date
    FROM app_events
    WHERE event_type = 'login'
    GROUP BY user_id
),
payment_stats AS (
    SELECT 
        user_id,
        SUM(CASE WHEN payment_status = 'failed' 
                      AND payment_date > date('2025-08-01', '-90 days') 
            THEN 1 ELSE 0 END) AS failed_payments_last_90d
    FROM payments
    GROUP BY user_id
),
ticket_stats AS (
    SELECT 
        user_id,
        COUNT(user_id) AS support_ticket_total
    FROM app_events
    WHERE event_type='support_ticket'
    GROUP BY user_id
)
SELECT
    u.user_id,
    u.plan_type,
    u.country,
    u.acquisition_channel,
    julianday('2025-08-01') - julianday(u.signup_date) AS tenure_days,
    COALESCE(ws.total_workouts,0) AS total_workouts,
    COALESCE(ws.avg_duration_minutes,0) AS avg_duration_minutes,
    COALESCE(ws.workouts_last_30d, 0) AS workouts_last_30d,
    COALESCE(julianday('2025-08-01') - julianday(ls.last_login_date), 9999) AS days_since_last_login,
    COALESCE(ps.failed_payments_last_90d, 0) AS failed_payments_last_90d,
    COALESCE(ts.support_ticket_total, 0) AS support_ticket_total,
    CASE WHEN s.status = 'cancelled' THEN 1 ELSE 0 END AS label
FROM users u
LEFT JOIN workout_stats ws ON u.user_id = ws.user_id
LEFT JOIN login_stats ls ON u.user_id = ls.user_id
LEFT JOIN payment_stats ps ON u.user_id = ps.user_id
LEFT JOIN ticket_stats ts ON u.user_id = ts.user_id
LEFT JOIN subscriptions s ON u.user_id = s.user_id 
""", conn)

,user_id,plan_type,country,acquisition_channel,tenure_days,total_workouts,avg_duration_minutes,workouts_last_30d,days_since_last_login,failed_payments_last_90d,support_ticket_total,label
0,1,Basic,UK,organic,464.0,0,0.000000,0,286.0,0,0,1
1,2,Premium,US,organic,382.0,25,32.120000,0,9.0,0,2,0
2,3,Basic,IN,referral,478.0,56,35.589286,9,7.0,0,2,0
3,4,Premium,IN,organic,186.0,3,27.333333,0,140.0,0,1,1
4,5,Basic,DE,referral,392.0,44,31.090909,6,1.0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
795,796,Premium,AU,influencer,439.0,19,31.947368,0,107.0,0,2,1
796,797,Elite,DE,social_ads,81.0,0,0.000000,0,52.0,0,0,1
797,798,Basic,US,social_ads,428.0,1,37.000000,0,366.0,0,0,1
798,799,Premium,US,referral,264.0,17,31.647059,0,0.0,0,2,0


Inside each CTE — just aggregate normally; don't worry about missing users yet
In the final SELECT — LEFT JOIN from users (which has everyone) onto each CTE, and use COALESCE(..., 0) to turn "no match" into "0"

You have to join from the complete table (users) onto the incomplete ones (the CTEs), not the other way around.

I used 9999 instead of 0 for `days_since_last_login`. 0 would mean "logged in today," which is the opposite of the truth for someone who's never logged in at all. A large number like 9999 signals "extremely long since last login" (or effectively "never"), which is the correct signal for a churn model — you want this feature to make these users look high risk, not low risk.

## From SQL to PyTorch

The SQL query gives you a table: one row per user, various numeric/text columns, plus a label. 

PyTorch doesn't understand pandas DataFrames — it only understands tensors. So we need to convert the table into that format, then train a small model on it.

The pipeline has five stages:

SQL query → pandas DataFrame → numpy arrays → PyTorch tensors → trained model

### Step 1: Get query into pandas

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("fitness_app.db")

df = pd.read_sql_query("""
WITH workout_stats AS (
    SELECT
        user_id,
        COUNT(user_id) AS total_workouts,
        AVG(duration_minutes) AS avg_duration_minutes,
        SUM(CASE WHEN workout_date > date('2025-08-01', '-30 days') THEN 1 ELSE 0 END) AS workouts_last_30d
    FROM workouts
    GROUP BY user_id
),
login_stats AS (
    SELECT
        user_id,
        MAX(event_date) AS last_login_date
    FROM app_events
    WHERE event_type = 'login'
    GROUP BY user_id
),
payment_stats AS (
    SELECT 
        user_id,
        SUM(CASE WHEN payment_status = 'failed' 
                      AND payment_date > date('2025-08-01', '-90 days') 
            THEN 1 ELSE 0 END) AS failed_payments_last_90d
    FROM payments
    GROUP BY user_id
),
ticket_stats AS (
    SELECT 
        user_id,
        COUNT(user_id) AS support_ticket_total
    FROM app_events
    WHERE event_type='support_ticket'
    GROUP BY user_id
)
SELECT
    u.user_id,
    u.plan_type,
    u.country,
    u.acquisition_channel,
    julianday('2025-08-01') - julianday(u.signup_date) AS tenure_days,
    COALESCE(ws.total_workouts,0) AS total_workouts,
    COALESCE(ws.avg_duration_minutes,0) AS avg_duration_minutes,
    COALESCE(ws.workouts_last_30d, 0) AS workouts_last_30d,
    COALESCE(julianday('2025-08-01') - julianday(ls.last_login_date), 9999) AS days_since_last_login,
    COALESCE(ps.failed_payments_last_90d, 0) AS failed_payments_last_90d,
    COALESCE(ts.support_ticket_total, 0) AS support_ticket_total,
    CASE WHEN s.status = 'cancelled' THEN 1 ELSE 0 END AS label
FROM users u
LEFT JOIN workout_stats ws ON u.user_id = ws.user_id
LEFT JOIN login_stats ls ON u.user_id = ls.user_id
LEFT JOIN payment_stats ps ON u.user_id = ps.user_id
LEFT JOIN ticket_stats ts ON u.user_id = ts.user_id
LEFT JOIN subscriptions s ON u.user_id = s.user_id 
""", conn)
df.shape  # should be (800, 12)

(800, 12)

In [9]:
df.head()

,user_id,plan_type,country,acquisition_channel,tenure_days,total_workouts,avg_duration_minutes,workouts_last_30d,days_since_last_login,failed_payments_last_90d,support_ticket_total,label
0,1,Basic,UK,organic,464.0,0,0.000000,0,286.0,0,0,1
1,2,Premium,US,organic,382.0,25,32.120000,0,9.0,0,2,0
2,3,Basic,IN,referral,478.0,56,35.589286,9,7.0,0,2,0
3,4,Premium,IN,organic,186.0,3,27.333333,0,140.0,0,1,1
4,5,Basic,DE,referral,392.0,44,31.090909,6,1.0,0,1,0


In the context of this project: 

Every user in `fitness_app.db` who has `status = 'cancelled'` in the `subscriptions` table has "churned" — they signed up, used the app for a while, and then quit. That's what the `label` column represents: `1` if they churned (cancelled), `0` if they're still an active subscriber.

### Step 2: Handle the text columns

The table has `plan_type`, `country`, `acquisition_channel` as text ('Basic', 'US', etc.).

PyTorch tensors are purely numeric — there's no way to feed the literal word "Basic" into a neural network.

The standard fix is one-hot encoding — turning one text column into several 0/1 columns, one per category:

In [10]:
df_model = pd.get_dummies(df, columns=["plan_type", "country", "acquisition_channel"], drop_first=True)
df_model.head()

,user_id,tenure_days,total_workouts,avg_duration_minutes,workouts_last_30d,days_since_last_login,failed_payments_last_90d,support_ticket_total,label,plan_type_Elite,...,country_BR,country_CA,country_DE,country_IN,country_UK,country_US,acquisition_channel_organic,acquisition_channel_paid_search,acquisition_channel_referral,acquisition_channel_social_ads
0,1,464.0,0,0.000000,0,286.0,0,0,1,False,...,False,False,False,False,True,False,True,False,False,False
1,2,382.0,25,32.120000,0,9.0,0,2,0,False,...,False,False,False,False,False,True,True,False,False,False
2,3,478.0,56,35.589286,9,7.0,0,2,0,False,...,False,False,False,True,False,False,False,False,True,False
3,4,186.0,3,27.333333,0,140.0,0,1,1,False,...,False,False,False,True,False,False,True,False,False,False
4,5,392.0,44,31.090909,6,1.0,0,1,0,False,...,False,False,True,False,False,False,False,False,True,False


Country had 7 unique values but only generated 6 columns (no country_AU). 

The drop_first=True — one category becomes the "default," represented by all zeros across its sibling columns. If a user were from Australia instead (the dropped category), all 6 country_* columns would read False, and that all-False pattern itself means "Australia" implicitly.

Including all n columns would create a mathematical redundancy (the model could always calculate the last category from the other columns being all zero), which can cause instability in some models. Dropping one avoids that.

### Step 3 - Separating features from the label, and converting to plain numeric arrays.

Every supervised ML model needs two things kept apart:

X — the inputs (everything the model is allowed to look at to make a prediction)\
y — the answer (what you're trying to predict — here, the label, whether they churned)

user_id needs to be excluded from X too. It's just meaningless: user_id is an arbitrary number assigned in whatever order users signed up. It has no real relationship to churn, and including it risks the model learning spurious patterns from something that's essentially random noise (e.g., "users with higher IDs churn more" — which would just be an artifact of who signed up later, not a real signal).

In [11]:
feature_cols = [c for c in df_model.columns if c not in ("user_id", "label")]
X = df_model[feature_cols].astype(float).values
y = df_model["label"].astype(float).values

print(X.shape, y.shape)
print(type(X), type(y))

(800, 19) (800,)
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


### Step 4: Train/test split

This matters a lot: if you train and evaluate a model on the exact same data, you have no way of knowing whether it actually learned something useful, or just memorized the answers.

The fix is to split the 800 users into two groups before training:

Train set (typically ~80%) — the model learns from these\
Test set (typically ~20%) — completely hidden from training, used only afterward to check "does this model work on users it's never seen?"

In [12]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(X))  # [0, 1, 2, ..., 799]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, indices, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(640, 19) (160, 19)
(640,) (160,)


`test_size=0.2` — 20% of users held out for testing, 80% for training.

`stratify=y` — It forces both the train and test sets to have roughly the same proportion of churned vs. non-churned users as the full dataset. Without it, random chance could hand you a test set with, say, 60% churners when your real data is only ~35% churned — and your evaluation results would be misleading.
`indices` - Notice `train_test_split` can split as many arrays as you hand it, all using the same shuffle — so idx_test tells you exactly which original row positions ended up in your test set, in the same order as `X_test/y_test/test_probs`.

In [13]:
print(y_train.mean(), y_test.mean())

0.3453125 0.34375


### Step 5: Scaling the features

The feature columns — tenure_days might range into the hundreds, total_workouts might be 0-100, but the one-hot encoded columns (plan_type_Elite, etc.) are only ever 0 or 1. Neural networks are sensitive to these scale differences: a feature that ranges in the hundreds can dominate the learning process purely because its numbers are bigger, not because it's actually more important. This can slow down or destabilize training.

The fix is to standarize by rescaling every numeric feature so it has mean 0 and standard deviation 1:

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.mean(axis=0).round(2))
print(X_train_scaled.std(axis=0).round(2))

[-0.  0. -0.  0. -0. -0.  0. -0.  0. -0. -0. -0. -0.  0. -0. -0.  0.  0.
 -0.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


scaler.fit_transform(X_train) — calculates the mean/std from the training data, then applies the rescaling\
scaler.transform(X_test) — applies that same training-derived mean/std to the test data, without recalculating

If you let the scaler "see" the test set while calculating its mean/std, you'd be leaking a small amount of information from data the model is supposed to never have seen.

### Step 6: numpy → PyTorch tensors

In [16]:
import torch

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(X_train_t.shape, y_train_t.shape)
print(X_test_t.shape, y_test_t.shape)

torch.Size([640, 19]) torch.Size([640, 1])
torch.Size([160, 19]) torch.Size([160, 1])


Neural networks do a huge number of matrix multiplications during training, and float32 is the standard precision PyTorch expects — without specifying it, numpy might hand over float64 (double precision) or even mixed integer/boolean types, which can cause PyTorch to throw a type-mismatch error.

The `y` array is currently 0s and 1s — but the loss function we'll use later expects floats.

`.unsqueeze(1)` inserts a new dimension at position 1, turning (640,) into (640, 1), without changing any of the actual values — just reshaping the container.

### Step 7: Building the neural network

In PyTorch, a model is a Python class that describes two things: what layers exist, and how data flows through them.

In [17]:
import torch.nn as nn

class ChurnClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

model = ChurnClassifier(n_features=X_train_t.shape[1])
print(model)

ChurnClassifier(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=1, bias=True)
  )
)


`nn.Linear(n_features, 32)` — this is the fundamental building block. It takes your 19 input features and combines them into 32 new numbers, using weights the model will learn during training. Concretely, each of the 32 outputs is a weighted sum of all 19 inputs (plus a bias term) — literally output = w1*x1 + w2*x2 + ... + w19*x19 + b, repeated 32 times with 32 different sets of weights. Those weights start random and get adjusted during training.

`nn.ReLU()` — this is the "non-linearity." Without it, stacking multiple Linear layers would mathematically collapse into being equivalent to just one linear layer, no matter how many you stack — because a chain of purely linear operations is still linear. ReLU breaks that by just doing max(0, x) — zeroing out negative values, passing positive ones through unchanged. That simple trick is what lets the network learn non-linear patterns (like "risk goes up when tenure is low AND workouts are low, but not when only one of those is true").

`nn.Dropout(0.2)` — during training, this randomly "turns off" 20% of the neurons at each step. It prevents the model from over-relying on any single feature combination — a common technique against overfitting (memorizing training data instead of learning general patterns).

`nn.Linear(16, 1)` — the final layer, collapsing everything down to a single number: the model's raw prediction for "how likely is this user to churn?"

`forward(self, x)` — defines the actual flow: take input x, push it through self.net in order, return the result. This method runs every time you call model(some_tensor).

### Step 8: Loss function and optimizer

Two more pieces before training can happen — the model can make predictions, but right now it has no way to know if those predictions are good or bad, or how to improve.

In [18]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(pos_weight)

tensor([1.8959])


`criterion` (the loss function) — this measures "how wrong was the prediction," which the model uses to adjust its weights.

`BCEWithLogitsLoss` is the standard choice for binary classification (churn / no churn) — "BCE" stands for Binary Cross-Entropy, a way of scoring probability-based predictions that penalizes confident-wrong predictions much more heavily than uncertain ones.

`pos_weight` — this is addressing something specific to the data: only ~35% of users churned, so the model could get "decent" accuracy just by predicting "never churns" for everyone. pos_weight tells the loss function to penalize missed churners (false negatives) more heavily than false alarms, correcting for that imbalance. The value is 1.9 (since $(1 - 0.345) / 0.345 \approx 1.9$)

`optimizer` — this is the algorithm that actually updates the model's weights based on the loss. Adam is a widely-used, well-behaved default choice. lr=1e-3 (the learning rate) controls how big each update step is — too high and training becomes unstable/erratic, too low and training crawls. weight_decay=1e-4 is a mild extra regularization technique (nudges weights toward staying small, another anti-overfitting measure, similar in spirit to Dropout).

### Step 9: The training loop

This is where the model actually learns.

In [19]:
EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, loss: {loss.item():.4f}")

Epoch 20, loss: 0.8917
Epoch 40, loss: 0.8367
Epoch 60, loss: 0.7451
Epoch 80, loss: 0.6052
Epoch 100, loss: 0.4533


`model.train()` — puts the model in "training mode." This matters specifically because of Dropout — you want neurons randomly zeroed out during training (for the regularization effect discussed earlier), but not during evaluation. This flag tells PyTorch which behavior to use.

`optimizer.zero_grad()` — PyTorch accumulates gradients by default rather than overwriting them. This line clears out any leftover gradient values from the previous loop iteration before computing new ones.

`logits = model(X_train_t)` — runs all 640 training users through the network at once, producing 640 raw predictions (called "logits" — unprocessed scores, not yet converted into 0-1 probabilities).

`loss = criterion(logits, y_train_t)` — compares those 640 predictions against the 640 real labels, producing a single number summarizing "how wrong was the model, on average, across everyone."

`loss.backward()` — this is the "learning" step. PyTorch automatically calculates the gradient — for every single weight in the network, how much would nudging that weight up or down change the loss? This uses calculus (the chain rule, applied automatically) to trace back through every layer.

`optimizer.step()` — actually applies those calculated gradients, nudging every weight slightly in the direction that reduces loss (scaled by the learning rate from before).


### Step 10: Evaluating on the test set

Here's the moment of truth — checking whether what the model learned actually generalizes to the 160 users it never saw during training, not just users it memorized.

In [20]:
from sklearn.metrics import roc_auc_score, classification_report

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits).numpy().ravel()
    test_preds = (test_probs > 0.5).astype(int)

print("Test AUC:", roc_auc_score(y_test, test_probs))
print(classification_report(y_test, test_preds))

Test AUC: 0.9402597402597402
              precision    recall  f1-score   support

         0.0       0.95      0.72      0.82       105
         1.0       0.64      0.93      0.76        55

    accuracy                           0.79       160
   macro avg       0.79      0.83      0.79       160
weighted avg       0.84      0.79      0.80       160



`model.eval()` — the opposite of `model.train()` from before. This turns off `Dropout` (you want the model's full, deterministic capacity when actually making real predictions, not randomly disabled neurons) and tells other layers that might behave differently in training vs. inference (e.g. batch norm) to switch modes too.

`with torch.no_grad():` — tells PyTorch "don't bother tracking gradients for anything inside this block." During training, PyTorch tracks every calculation so it can later compute gradients via `.backward()`. During evaluation, tracking gradients would just waste memory and computation for no benefit.

`torch.sigmoid(test_logits)` — The model's raw output (`logits`) is an unbounded number, not a 0-1 probability. `sigmoid` is the standard function that squashes any real number into the range (0, 1) — turning "raw score" into "predicted probability of churning."

`test_probs > 0.5` — converts probabilities into hard yes/no predictions using 0.5 as the decision threshold (above 0.5 → predict "will churn").

`roc_auc_score` — a metric specifically well-suited for imbalanced classification like this. It measures how well the model ranks churners above non-churners across every possible threshold, not just the 0.5 threshold. A score of 0.5 = random guessing, 1.0 = perfect separation.

The classification_report shows precision/recall/F1 broken down by class, which tells a richer story than accuracy alone.

A 0.94 AUC means the model is doing an excellent job ranking who's more likely to churn versus who isn't, well above the 0.5 "random guessing" baseline.


Precision: Measures the accuracy of positive predictions (True Positives / (True Positives + False Positives)).\
Out of all instances predicted as a specific class, how many actually belonged to it?

Recall: Measures the ability to find all positive instances (True Positives / (True Positives + False Negatives)).\
Out of all actual instances of a class, how many did the model find?

Accuracy: The overall ratio of correctly predicted instances across the entire test set.

Support: The actual number of true instances (occurrences) for that specific class in your dataset.

Class 1.0 (churners) — recall = 0.93: of the 55 users who actually churned, the model correctly flagged 93% of them. That's the direct payoff of the pos_weight correction from earlier — told the loss function "missing a churner is costly," and the model listened. For a retention team, this is usually the number that matters most: you'd rather over-flag some safe users than miss people who are actually about to leave.

Class 1.0 — precision = 0.64: but here's the trade-off — of everyone the model flagged as a churner, only 64% actually churned. The other 36% were false alarms.\
This is the direct cost of prioritizing recall so heavily: casting a wider net catches more real churners, but also catches more people who were fine.

Class 0.0 (non-churners) — recall = 0.72: of users who stayed, the model correctly identified 72% of them as safe — the other 28% got incorrectly flagged as at-risk (these are the false alarms feeding into that 0.64 precision above).

Class 0.0 — precision = 0.95: but here's the trade-off — of everyone the model flagged as a non-churner, only 95% actually non-churned. The other 5% were false alarms.

The trade-offs depend entirely on the business cost of each mistake.

If reaching out to a safe user costs almost nothing (a cheap email), erring toward high recall (catch almost all real churners, tolerate some false alarms) is the right call — which is exactly what this model is doing. 

If outreach were expensive (a phone call from a human retention specialist), you'd want to shift the threshold to favor precision instead. 

The "right" threshold isn't a fixed 0.5 — it depends on the relative cost of false positives vs. false negatives, something a purely technical accuracy number can't tell you.

### Step 11: Back to a business number: expected revenue at risk

Leadership doesn't just want "will they churn"; they want expected revenue at risk, so retention can prioritise who to contact first. Need to map these probabilities back to actual `user_id`s and dollar amounts.

In [21]:
results = df.iloc[idx_test][["user_id", "plan_type", "country"]].copy()
results["churn_probability"] = test_probs

results.head()

,user_id,plan_type,country,churn_probability
544,545,Basic,CA,0.832518
784,785,Basic,AU,0.055413
334,335,Premium,US,0.137915
366,367,Basic,UK,0.566398
656,657,Basic,AU,0.131223


`df.iloc[idx_test]` — this is the key line. `.iloc` selects rows by their raw position number (not by any column value), and `idx_test` is exactly the list of original row positions your test set came from — the thing you preserved a few steps ago specifically to make this possible. This pulls out the same 160 users, in the same order, as `test_probs` — so row 0 of `results` now correctly corresponds to row 0 of `test_probs`.

`.copy()` — a small but important habit: without it, pandas sometimes gives you a "view" into the original `df` rather than an independent copy, and modifying `results` afterward can trigger a confusing warning, or in rare cases even alter `df` unintentionally. Copying first avoids that ambiguity.

`results["churn_probability"] = test_probs` — attaches your model's predictions as a new column, aligned row-by-row since both are in the same order now.

Probability alone doesn't tell the retention team who to prioritize — 

a 90\%-likely-to-churn Basic plan user (9.99/month) might matter less than a 40\%-likely-to-churn Elite user ($29.99/month).

The business-relevant number is probability × price, since that's the actual expected dollar loss.

In [22]:
plan_price = {"Basic": 9.99, "Premium": 19.99, "Elite": 29.99}
results["monthly_price"] = results["plan_type"].map(plan_price)
results["expected_revenue_at_risk"] = results["churn_probability"] * results["monthly_price"]

top_at_risk = results.sort_values("expected_revenue_at_risk", ascending=False).head(10)
print(top_at_risk.to_string(index=False))

 user_id plan_type country  churn_probability  monthly_price  expected_revenue_at_risk
     247     Elite      US           0.846529          29.99                 25.387401
     235     Elite      UK           0.669521          29.99                 20.078934
     566   Premium      BR           0.999787          19.99                 19.985740
     293     Elite      IN           0.632801          29.99                 18.977705
      74     Elite      UK           0.614728          29.99                 18.435697
     734   Premium      UK           0.919133          19.99                 18.373475
      76     Elite      IN           0.611651          29.99                 18.343410
     170   Premium      US           0.904962          19.99                 18.090181
     747   Premium      IN           0.904120          19.99                 18.073367
      92     Elite      UK           0.601289          29.99                 18.032649


`.map(plan_price)` — this is a clean way to convert a category `('Basic', 'Premium', 'Elite')` into a corresponding number, using a Python dictionary as a lookup table. For every row, it looks up that row's `plan_type` in `plan_price` and returns the matching value.

`churn_probability * monthly_price` — If there's a 93\% chance of losing 10/month, the expected loss is 9.30 — not the full 10 (since there's some chance they stay) and not 0 (since there's real risk). Multiplying probability by consequence is how you turn a prediction into a decision-relevant number.

`.sort_values(..., ascending=False).head(10)` — ranks users by that expected-loss number, highest first, giving you a genuine priority list rather than just a probability list.

This table is exactly the kind of output a retention team would actually act on: 

`user 247` with a 84.66\% churn probability on a $29.99 Elite plan sits at the top, correctly outranking `user 566` who has an even higher probability (99.98\%) but on a cheaper plan — because the model is now optimising for dollars at risk, not just likelihood.